# 如何评估摘要任务

在本 notebook 中，我们通过一个简单的示例深入探讨生成式摘要（abstractive summarization）任务的评估技术。我们将探讨 [ROUGE](https://aclanthology.org/W04-1013/) 和 [BERTScore](https://arxiv.org/abs/1904.09675) 等传统评估方法，并展示一种使用 LLM 作为评估器的较新方法。

评估摘要质量是一个耗时的过程，因为它涉及连贯性、简洁性、可读性和内容等不同的质量指标。`ROUGE`、`BERTScore` 等传统自动评估指标具体且可靠，但它们可能与摘要的实际质量相关性不高。它们与人工判断的相关性相对较低，尤其是在开放式生成任务上（[Liu et al., 2023](https://arxiv.org/pdf/2303.16634.pdf)）。人们越来越需要依赖人工评估、用户反馈或基于模型的指标，同时警惕潜在的偏差。虽然人工判断能提供极有价值的洞见，但它往往难以规模化，且成本可能高得令人却步。

除了这些传统指标之外，我们还展示了一种方法（[G-Eval](https://arxiv.org/pdf/2303.16634.pdf)），它利用大语言模型（LLM）作为一种新颖的、无需参考答案的指标来评估生成式摘要。在本例中，我们使用 `gpt-4` 为候选输出打分。`gpt-4` 已经有效地学习到了一个关于语言质量的内部模型，使它能够区分流畅、连贯的文本和低质量文本。利用这种内部评分机制，就可以自动评估由 LLM 生成的新候选输出。


## 设置


In [ ]:
# Installing necessary packages for the evaluation
# rouge: For evaluating with ROUGE metric
# bert_score: For evaluating with BERTScore
# openai: To interact with OpenAI's API
!pip install rouge --quiet
!pip install bert_score --quiet
!pip install openai --quiet

In [23]:
from openai import OpenAI
import os
import re
import pandas as pd

# Python Implementation of the ROUGE Metric
from rouge import Rouge

# BERTScore leverages the pre-trained contextual embeddings from BERT and matches words in candidate and reference sentences by cosine similarity.
from bert_score import BERTScorer

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))


<IPython.core.display.Javascript object>

## 示例任务

在本 notebook 中，我们将使用下面的摘要示例。请注意，我们提供了两份待比较的生成摘要，以及一份人工撰写的参考摘要，`ROUGE` 和 `BERTScore` 等评估指标需要用到参考摘要。

原文节选（`excerpt`）：

> OpenAI's mission is to ensure that artificial general intelligence (AGI) benefits all of humanity. OpenAI will build safe and beneficial AGI directly, but will also consider its mission fulfilled if its work aids others to achieve this outcome. OpenAI follows several key principles for this purpose. First, broadly distributed benefits - any influence over AGI's deployment will be used for the benefit of all, and to avoid harmful uses or undue concentration of power. Second, long-term safety - OpenAI is committed to doing the research to make AGI safe, and to promote the adoption of such research across the AI community. Third, technical leadership - OpenAI aims to be at the forefront of AI capabilities. Fourth, a cooperative orientation - OpenAI actively cooperates with other research and policy institutions, and seeks to create a global community working together to address AGI's global challenges.

摘要：

| 参考摘要 /`ref_summary`（人工撰写）                                                                                                                                                                                                                                                                                                                         | 评估摘要 1 / `eval_summary_1`（系统生成）                                                                                                                                                                                                                                                                                                                               | 评估摘要 2 / `eval_summary_2`（系统生成）                                                                                                                                                                                                                                                   |
| -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| OpenAI aims to ensure artificial general intelligence (AGI) is used for everyone's benefit, avoiding harmful uses or undue power concentration. It is committed to researching AGI safety, promoting such studies among the AI community. OpenAI seeks to lead in AI capabilities and cooperates with global research and policy institutions to address AGI's challenges. | OpenAI aims to AGI benefits all humanity, avoiding harmful uses and power concentration. It pioneers research into safe and beneficial AGI and promotes adoption globally. OpenAI maintains technical leadership in AI while cooperating with global institutions to address AGI challenges. It seeks to lead a collaborative worldwide effort developing AGI for collective good. | OpenAI aims to ensure AGI is for everyone's use, totally avoiding harmful stuff or big power concentration. Committed to researching AGI's safe side, promoting these studies in AI folks. OpenAI wants to be top in AI things and works with worldwide research, policy groups to figure AGI's stuff. |

花点时间想一想，你个人更喜欢哪份摘要，以及哪份摘要真正很好地概括了 OpenAI 的使命。


In [9]:
excerpt = "OpenAI's mission is to ensure that artificial general intelligence (AGI) benefits all of humanity. OpenAI will build safe and beneficial AGI directly, but will also consider its mission fulfilled if its work aids others to achieve this outcome. OpenAI follows several key principles for this purpose. First, broadly distributed benefits - any influence over AGI's deployment will be used for the benefit of all, and to avoid harmful uses or undue concentration of power. Second, long-term safety - OpenAI is committed to doing the research to make AGI safe, and to promote the adoption of such research across the AI community. Third, technical leadership - OpenAI aims to be at the forefront of AI capabilities. Fourth, a cooperative orientation - OpenAI actively cooperates with other research and policy institutions, and seeks to create a global community working together to address AGI's global challenges."
ref_summary = "OpenAI aims to ensure artificial general intelligence (AGI) is used for everyone's benefit, avoiding harmful uses or undue power concentration. It is committed to researching AGI safety, promoting such studies among the AI community. OpenAI seeks to lead in AI capabilities and cooperates with global research and policy institutions to address AGI's challenges."
eval_summary_1 = "OpenAI aims to AGI benefits all humanity, avoiding harmful uses and power concentration. It pioneers research into safe and beneficial AGI and promotes adoption globally. OpenAI maintains technical leadership in AI while cooperating with global institutions to address AGI challenges. It seeks to lead a collaborative worldwide effort developing AGI for collective good."
eval_summary_2 = "OpenAI aims to ensure AGI is for everyone's use, totally avoiding harmful stuff or big power concentration. Committed to researching AGI's safe side, promoting these studies in AI folks. OpenAI wants to be top in AI things and works with worldwide research, policy groups to figure AGI's stuff."

<IPython.core.display.Javascript object>

## 使用 ROUGE 进行评估

[ROUGE](https://aclanthology.org/W04-1013/) 是 Recall-Oriented Understudy for Gisting Evaluation 的缩写，主要衡量生成输出与参考文本之间的词语重叠程度。它是评估自动摘要任务的常用指标。在其各种变体中，`ROUGE-L` 可以揭示系统生成摘要与参考摘要之间的最长连续匹配，衡量系统在多大程度上保留了原始摘要的精髓。


In [10]:
# function to calculate the Rouge score
def get_rouge_scores(text1, text2):
    rouge = Rouge()
    return rouge.get_scores(text1, text2)


rouge_scores_out = []

# Calculate the ROUGE scores for both summaries using reference
eval_1_rouge = get_rouge_scores(eval_summary_1, ref_summary)
eval_2_rouge = get_rouge_scores(eval_summary_2, ref_summary)

for metric in ["rouge-1", "rouge-2", "rouge-l"]:
    for label in ["F-Score"]:
        eval_1_score = eval_1_rouge[0][metric][label[0].lower()]
        eval_2_score = eval_2_rouge[0][metric][label[0].lower()]

        row = {
            "Metric": f"{metric} ({label})",
            "Summary 1": eval_1_score,
            "Summary 2": eval_2_score,
        }
        rouge_scores_out.append(row)


def highlight_max(s):
    is_max = s == s.max()
    return [
        "background-color: lightgreen" if v else "background-color: white"
        for v in is_max
    ]


rouge_scores_out = (
    pd.DataFrame(rouge_scores_out)
    .set_index("Metric")
    .style.apply(highlight_max, axis=1)
)

rouge_scores_out

,Summary 1,Summary 2
Metric,,
rouge-1 (F-Score),0.488889,0.511628
rouge-2 (F-Score),0.230769,0.163265
rouge-l (F-Score),0.488889,0.511628


<IPython.core.display.Javascript object>

该表展示了以参考文本为基准评估两份不同摘要的 `ROUGE` 分数。在 `rouge-1` 上，摘要 2 优于摘要 1，表明其单个词语的重叠度更好；在 `rouge-l` 上，摘要 2 的分数也更高，意味着在最长公共子序列上匹配更接近，因此在捕捉原文主要内容和顺序方面可能是更好的整体摘要。由于摘要 2 中有许多词语和短语直接摘自原文节选，它与参考摘要的重叠度可能更高，从而导致更高的 `ROUGE` 分数。

虽然 `ROUGE` 以及 [BLEU](https://aclanthology.org/P02-1040.pdf) 和 [METEOR](https://www.cs.cmu.edu/~alavie/METEOR/) 等类似指标提供了量化衡量，但它们往往无法捕捉一份优质摘要的真正精髓。它们与人工评分的相关性也较差。鉴于 LLM 的进步——它们擅长生成流畅、连贯的摘要——`ROUGE` 等传统指标可能会在无意中惩罚这些模型。当摘要的表述方式不同、但仍准确概括了核心信息时，尤其如此。


## 使用 BERTScore 进行评估

ROUGE 依赖于预测文本和参考文本中词语的精确出现，无法理解其背后的语义。这正是 [BERTScore](https://arxiv.org/abs/1904.09675) 的用武之地：它利用 BERT 模型的上下文嵌入，旨在评估机器生成文本场景下预测句子与参考句子之间的相似度。通过比较两个句子的嵌入，`BERTScore` 能够捕捉传统基于 n-gram 的指标可能遗漏的语义相似性。


In [11]:
# Instantiate the BERTScorer object for English language
scorer = BERTScorer(lang="en")

# Calculate BERTScore for the summary 1 against the excerpt
# P1, R1, F1_1 represent Precision, Recall, and F1 Score respectively
P1, R1, F1_1 = scorer.score([eval_summary_1], [ref_summary])

# Calculate BERTScore for summary 2 against the excerpt
# P2, R2, F2_2 represent Precision, Recall, and F1 Score respectively
P2, R2, F2_2 = scorer.score([eval_summary_2], [ref_summary])

print("Summary 1 F1 Score:", F1_1.tolist()[0])
print("Summary 2 F1 Score:", F2_2.tolist()[0])

Some weights of the model checkpoint at roberta-large were not used when initializing RobertaModel: ['lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.decoder.weight', 'lm_head.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Summary 1 F1 Score: 0.9227314591407776
Summary 2 F1 Score: 0.9189572930335999


<IPython.core.display.Javascript object>

两份摘要的 F1 分数非常接近，表明它们在捕捉关键信息方面可能表现相近。不过，对这一微小差异的解读应保持谨慎。由于 `BERTScore` 可能无法完全把握人类评估者能够理解的细微之处和高层概念，仅依赖这一指标可能会误判摘要的实际质量和细节。将 `BERTScore` 与人工判断及其他指标相结合的综合方法，可以提供更可靠的评估。


## 使用 GPT-4 进行评估

这里我们使用 `gpt-4` 实现一个**无需参考答案**的文本评估器示例，其灵感来自 [G-Eval](https://arxiv.org/pdf/2303.16634.pdf) 框架，该框架使用大语言模型评估生成文本的质量。与依赖参考摘要进行比较的 `ROUGE` 或 `BERTScore` 等指标不同，基于 `gpt-4` 的评估器仅根据输入提示和文本来评估生成内容的质量，无需任何真实参考。这使它适用于人工参考稀少或不可用的新数据集和新任务。 

该方法概述如下：

1. 我们定义四个不同的评估标准：
    1. **相关性（Relevance）**：评估摘要是否只包含重要信息并排除了冗余内容。
    2. **连贯性（Coherence）**：评估摘要的逻辑流程和组织结构。
    3. **一致性（Consistency）**：检查摘要是否与源文档中的事实一致。
    4. **流畅性（Fluency）**：评价摘要的语法和可读性。
2. 我们为每个标准精心设计提示，以原始文档和摘要作为输入，利用思维链生成，并引导模型为每个标准输出 1-5 的数值分数。 
3. 我们使用定义好的提示让 `gpt-4` 生成分数，并在各份摘要之间进行比较。

在本演示中，我们使用直接评分函数，由 `gpt-4` 为每个指标生成一个离散分数（1-5）。对分数进行归一化并取加权和，可以得到更稳健、连续的分数，从而更好地反映摘要的质量和多样性。

In [15]:
# Evaluation prompt template based on G-Eval
EVALUATION_PROMPT_TEMPLATE = """
You will be given one summary written for an article. Your task is to rate the summary on one metric.
Please make sure you read and understand these instructions very carefully. 
Please keep this document open while reviewing, and refer to it as needed.

Evaluation Criteria:

{criteria}

Evaluation Steps:

{steps}

Example:

Source Text:

{document}

Summary:

{summary}

Evaluation Form (scores ONLY):

- {metric_name}
"""

# Metric 1: Relevance

RELEVANCY_SCORE_CRITERIA = """
Relevance(1-5) - selection of important content from the source. \
The summary should include only important information from the source document. \
Annotators were instructed to penalize summaries which contained redundancies and excess information.
"""

RELEVANCY_SCORE_STEPS = """
1. Read the summary and the source document carefully.
2. Compare the summary to the source document and identify the main points of the article.
3. Assess how well the summary covers the main points of the article, and how much irrelevant or redundant information it contains.
4. Assign a relevance score from 1 to 5.
"""

# Metric 2: Coherence

COHERENCE_SCORE_CRITERIA = """
Coherence(1-5) - the collective quality of all sentences. \
We align this dimension with the DUC quality question of structure and coherence \
whereby "the summary should be well-structured and well-organized. \
The summary should not just be a heap of related information, but should build from sentence to a\
coherent body of information about a topic."
"""

COHERENCE_SCORE_STEPS = """
1. Read the article carefully and identify the main topic and key points.
2. Read the summary and compare it to the article. Check if the summary covers the main topic and key points of the article,
and if it presents them in a clear and logical order.
3. Assign a score for coherence on a scale of 1 to 5, where 1 is the lowest and 5 is the highest based on the Evaluation Criteria.
"""

# Metric 3: Consistency

CONSISTENCY_SCORE_CRITERIA = """
Consistency(1-5) - the factual alignment between the summary and the summarized source. \
A factually consistent summary contains only statements that are entailed by the source document. \
Annotators were also asked to penalize summaries that contained hallucinated facts.
"""

CONSISTENCY_SCORE_STEPS = """
1. Read the article carefully and identify the main facts and details it presents.
2. Read the summary and compare it to the article. Check if the summary contains any factual errors that are not supported by the article.
3. Assign a score for consistency based on the Evaluation Criteria.
"""

# Metric 4: Fluency

FLUENCY_SCORE_CRITERIA = """
Fluency(1-3): the quality of the summary in terms of grammar, spelling, punctuation, word choice, and sentence structure.
1: Poor. The summary has many errors that make it hard to understand or sound unnatural.
2: Fair. The summary has some errors that affect the clarity or smoothness of the text, but the main points are still comprehensible.
3: Good. The summary has few or no errors and is easy to read and follow.
"""

FLUENCY_SCORE_STEPS = """
Read the summary and evaluate its fluency based on the given criteria. Assign a fluency score from 1 to 3.
"""


def get_geval_score(
    criteria: str, steps: str, document: str, summary: str, metric_name: str
):
    prompt = EVALUATION_PROMPT_TEMPLATE.format(
        criteria=criteria,
        steps=steps,
        metric_name=metric_name,
        document=document,
        summary=summary,
    )
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=5,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
    )
    return response.choices[0].message.content


evaluation_metrics = {
    "Relevance": (RELEVANCY_SCORE_CRITERIA, RELEVANCY_SCORE_STEPS),
    "Coherence": (COHERENCE_SCORE_CRITERIA, COHERENCE_SCORE_STEPS),
    "Consistency": (CONSISTENCY_SCORE_CRITERIA, CONSISTENCY_SCORE_STEPS),
    "Fluency": (FLUENCY_SCORE_CRITERIA, FLUENCY_SCORE_STEPS),
}

summaries = {"Summary 1": eval_summary_1, "Summary 2": eval_summary_2}

data = {"Evaluation Type": [], "Summary Type": [], "Score": []}

for eval_type, (criteria, steps) in evaluation_metrics.items():
    for summ_type, summary in summaries.items():
        data["Evaluation Type"].append(eval_type)
        data["Summary Type"].append(summ_type)
        result = get_geval_score(criteria, steps, excerpt, summary, eval_type)
        score_num = int(result.strip())
        data["Score"].append(score_num)

pivot_df = pd.DataFrame(data, index=None).pivot(
    index="Evaluation Type", columns="Summary Type", values="Score"
)
styled_pivot_df = pivot_df.style.apply(highlight_max, axis=1)
display(styled_pivot_df)

Summary Type,Summary 1,Summary 2
Evaluation Type,,
Coherence,5,3
Consistency,5,5
Fluency,3,2
Relevance,5,4


<IPython.core.display.Javascript object>


总体而言，摘要 1 在四个类别中的三个（连贯性、相关性和流畅性）上似乎都优于摘要 2。两份摘要都被判定为具有一致性。该结果可能表明，根据给定的评估标准，摘要 1 总体上更可取。

### 局限性

请注意，基于 LLM 的指标可能存在偏差，倾向于偏好 LLM 生成的文本而非人工撰写的文本。此外，基于 LLM 的指标对系统消息/提示很敏感。我们建议尝试其他有助于提升性能和/或获得一致分数的技术，在高质量但昂贵的评估与自动化评估之间取得适当平衡。还值得注意的是，这种评分方法目前受限于 `gpt-4` 的上下文窗口。

## 结论

生成式摘要的评估仍是一个有待进一步改进的开放领域。`ROUGE`、`BLEU` 和 `BERTScore` 等传统指标提供了有用的自动评估，但在捕捉语义相似性以及摘要质量的细微方面存在局限。此外，它们需要参考输出，而收集/标注参考输出的成本可能很高。基于 LLM 的指标有望成为一种无需参考答案、评估连贯性、流畅性和相关性的方法。然而，它们同样存在偏好 LLM 生成文本的潜在偏差。归根结底，要可靠地评估生成式摘要系统，最理想的是将自动指标与人工评估相结合。虽然人工评估对于全面理解摘要质量不可或缺，但它应辅以自动化评估，以实现高效、大规模的测试。该领域将继续发展出更稳健的评估技术，在质量、可扩展性和公平性之间取得平衡。推进评估方法对于推动生产应用的进步至关重要。

## 参考文献

- [G-EVAL: NLG Evaluation Using GPT-4 with Better Human Alignment](https://arxiv.org/pdf/2303.16634.pdf) - Liu Y, Iter D, Xu Y, Wang S, Xu R, Zhu C. 2023 年 5 月发表。
- [BERTScore: Evaluating Text Generation with BERT](https://arxiv.org/abs/1904.09675) - Zhang T, Kishore V, Wu F, Weinberger KQ, Artzi Y. 2020 年 2 月在线发表。
- [ROUGE: A Package for Automatic Evaluation of Summaries](https://aclanthology.org/W04-1013/) - Lin CY. 2004 年 7 月发表。
- [SummEval: Re-evaluating Summarization Evaluation](https://aclanthology.org/2021.tacl-1.24) - Fabbri et al. 2021 年 4 月发表。